In [1]:
from paths import *
from nano_maker.nanomaker import NanoMaker
from nano_maker.container.configs import (skeleton_weight_filename, naanobot_weight_filename,
                                          skeleton_config, naanobot_config, radial_config)

In [2]:
skeleton_weight_filename = skeleton_weight_filename
skeleton_cfg = skeleton_config
naanobot_weight_filename = naanobot_weight_filename
naanobot_config = naanobot_config
radial_cfg = radial_config

In [3]:
model = NanoMaker(skeleton_weight_filename=skeleton_weight_filename,
                  naanobot_weight_filename=naanobot_weight_filename,
                  skeleton_cfg=skeleton_config,
                  naanobot_config=naanobot_config,
                  radial_cfg=radial_cfg)

In [13]:
drug_i_want_to_deliver = "C[N+]1(C)C[C@@H]2C[C@H]1CN2c1ccc(nn1)-c1ccccc1 |r|"

In [14]:
model.ingest_chemical(drug_i_want_to_deliver)

Chemical Ingested: C[N+]1(C)C[C@@H]2C[C@H]1CN2c1ccc(nn1)-c1ccccc1 |r|
Drug Likeness Rules Passed: 4 / 4


In [15]:
pocket_data = model.generate_pocket_data(sampling_temp=0.3)
print(len(pocket_data))
print(type(pocket_data))

4
<class 'dict'>


In [16]:
pocket_data

{'SMILES': 'C[N+]1(C)C[C@@H]2C[C@H]1CN2c1ccc(nn1)-c1ccccc1 |r|',
 'Sampling_temperature': 0.3,
 '3D_skeleton': [[14.857084087572133, 6.3465736636404975, -1.7033587643581187],
  [15.154230152303894, 0.15616921351450697, 0.08656090763803403],
  [14.211074264977048, -1.5471795174689436, -2.790840120573979],
  [13.601423726828736, 3.6447717049335804, -0.16362831191176055],
  [12.500151925403536, -4.988758088332742, -1.6969413809966591],
  [11.415751954517546, -6.7643790464412215, -0.43949296605501803],
  [12.106629249141763, -4.286684991239511, -1.095599734093069],
  [10.582686969863103, -5.961591576104975, -3.5182171898450085],
  [6.571789518701785, -7.89504665258439, -6.546871366227119],
  [9.41317715173034, -0.5409312019142088, -6.939152477320565],
  [9.542583004955821, -4.575831780653288, -4.2615961462700716],
  [8.840939473362914, 6.765835808982602, 0.5890670393624586],
  [10.360529255944236, -0.8479005644138238, -1.783533907859089],
  [7.912205598130767, 3.32049599026224, -5.41172170

In [17]:
smiles_str = pocket_data["SMILES"]
skeleton = pocket_data['3D_skeleton']
aa_seq = pocket_data['aa_ids']

In [18]:
# GENERATION QUALITY ASSESSMENTS

In [19]:
import torch
import torch.nn.functional as F
from nano_maker.naanobot import NAAnoBot
from nano_maker.modules.nAAno.smiles_handler import smiles_fingerprint

@torch.no_grad()
def diagnose_generation(model, map4_enc, sph_coordinates, n=50, sampling_temperature=1.0, verbose=True):
    # explicitly force eval + disable dropout
    model.eval()
    for module in model.modules():
        if isinstance(module, torch.nn.Dropout):
            module.p = 0.0

    map4_enc = map4_enc.to(next(model.parameters()).device)

    bioch_context = [model.naano_module.get_nAAnovector("VOID") for _ in range(model.block_size)]
    coord_context = [[model.max_angstroms, 0, 0] for _ in range(model.block_size)]

    aa_chain = ""
    position_log = []  # track full run for post-hoc analysis

    for i, coordinate in enumerate(sph_coordinates[:n]):
        naano_X = model.naano_module.get_nAAno_X(
            coord_context, bioch_context, coordinate
        ).unsqueeze(0).to(map4_enc.device)

        output, _ = model.forward(naano_X, map4_enc)
        vector = output[0].detach().float()

        # distances to all amino acids
        distances = {}
        for aa_id, n_v in model.naano_module.nAAno_vectors.items():
            if aa_id == 'VOID':
                continue
            n_v_t = torch.tensor(n_v, dtype=torch.float32)
            distances[aa_id] = F.mse_loss(vector, n_v_t).item()

        sorted_distances = sorted(distances.items(), key=lambda x: x[1])
        top_aa, top_dist = sorted_distances[0]

        if verbose:
            print(f"\nPosition {i} | coord: ({coordinate[0]:.2f}Å, {coordinate[1]:.3f}rad, {coordinate[2]:.3f}rad)")
            print(f"  raw output:  {vector.numpy().round(2)}")
            print(f"  top 5:       {sorted_distances[:5]}")
            print(f"  best match:  {top_aa} (mse={top_dist:.4f})")
            print(f"  temperature: {sampling_temperature}")

        # stochastic selection at specified temperature
        aa_id = model.naano_module.approx_id(output[0], sampling_temperature=sampling_temperature)

        # flag if sampling diverged from greedy best
        if verbose and aa_id != top_aa:
            print(f"  sampled:     {aa_id} (diverged from greedy {top_aa})")

        bioch_context = bioch_context[1:] + [model.naano_module.get_nAAnovector(aa_id)]
        coord_context = coord_context[1:] + [coordinate]
        aa_chain += aa_id

        position_log.append({
            "position": i,
            "coordinate": coordinate,
            "greedy": top_aa,
            "sampled": aa_id,
            "top5": sorted_distances[:5],
            "diverged": aa_id != top_aa
        })

    if verbose:
        print(f"\nFinal chain: {aa_chain}")
        divergences = sum(1 for p in position_log if p["diverged"])
        print(f"Greedy/sample divergences: {divergences}/{len(position_log)} positions ({100*divergences/len(position_log):.1f}%)")

    return aa_chain, position_log


map4_enc = torch.tensor(smiles_fingerprint(drug_i_want_to_deliver), dtype=torch.float32).unsqueeze(0)
nb_cfg = naanobot_config.copy()
sph_coordinates = model._pocket_sph_skeleton()
naanobot_model = NAAnoBot(n_embd=nb_cfg["n_embd"], n_head=nb_cfg["n_head"],
                                           n_layers=nb_cfg["n_layers"],
                                           block_size=nb_cfg["block_size"],
                                           map4_res=nb_cfg["map4_res"],
                                           n_spatial_features=nb_cfg["n_spatial_features"],
                                           max_angstroms=nb_cfg["max_angstroms"],
                                           dropout=nb_cfg["dropout"],
                          loss_temperature=nb_cfg["loss_temperature"])
diagnose_generation(model=naanobot_model, map4_enc=map4_enc, sph_coordinates=sph_coordinates)


Position 0 | coord: (16.25Å, -0.857rad, 1.639rad)
  raw output:  [-0.14 -0.09 -0.23  0.64  0.23  0.35  0.86 -0.52  0.15 -0.74 -0.19  0.34
  0.71 -0.    0.48 -0.17 -0.39  0.15  0.2  -0.09 -0.13  0.9  -0.12 -0.03
  0.81 -0.8  -0.52 -0.06 -0.44  0.04 -0.14]
  top 5:       [('F', 0.350006639957428), ('V', 0.3528898060321808), ('I', 0.35482272505760193), ('L', 0.36175644397735596), ('T', 0.38458624482154846)]
  best match:  F (mse=0.3500)
  temperature: 1.0
  sampled:     A (diverged from greedy F)

Position 1 | coord: (15.59Å, -1.246rad, 1.333rad)
  raw output:  [-0.14  0.23 -0.05  0.47  0.31  0.26  0.93 -0.47 -0.11 -0.73 -0.47  0.37
  0.42 -0.14  0.31 -0.13 -0.43  0.15  0.21  0.11 -0.29  0.86  0.03 -0.34
  0.64 -1.1  -0.94 -0.09 -0.43  0.48 -0.18]
  top 5:       [('V', 0.4495181739330292), ('I', 0.45694032311439514), ('F', 0.46101564168930054), ('L', 0.46613502502441406), ('T', 0.46872833371162415)]
  best match:  V (mse=0.4495)
  temperature: 1.0
  sampled:     Y (diverged from greedy V

('AYIPQKPEETFPGTCQMFALKKMR',
 [{'position': 0,
   'coordinate': [16.24517250061035, -0.8572608232498169, 1.6391377449035645],
   'greedy': 'F',
   'sampled': 'A',
   'top5': [('F', 0.350006639957428),
    ('V', 0.3528898060321808),
    ('I', 0.35482272505760193),
    ('L', 0.36175644397735596),
    ('T', 0.38458624482154846)],
   'diverged': True},
  {'position': 1,
   'coordinate': [15.59134578704834, -1.2457653284072876, 1.3328397274017334],
   'greedy': 'V',
   'sampled': 'Y',
   'top5': [('V', 0.4495181739330292),
    ('I', 0.45694032311439514),
    ('F', 0.46101564168930054),
    ('L', 0.46613502502441406),
    ('T', 0.46872833371162415)],
   'diverged': True},
  {'position': 2,
   'coordinate': [14.54641342163086, -1.4385308027267456, 1.645592212677002],
   'greedy': 'V',
   'sampled': 'I',
   'top5': [('V', 0.45266222953796387),
    ('I', 0.45954087376594543),
    ('L', 0.4682050943374634),
    ('F', 0.46912094950675964),
    ('T', 0.4779234826564789)],
   'diverged': True},
  {